# EOD discount curve — par yield, zero rate, discount factors

SQLite → `numeraire_viz` → matplotlib.

Pipeline in DB:
- **`par_curve_*`** — FRED par yields (`quoted_rate`)
- **`discount_curve_*`** — bootstrapped zero rates + discount factors

DB path: repo `db.sqlite3` or `NUMERAIRE_DB_PATH` / `.env` (relative paths from **repo root**).

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd

from numeraire_viz import (
    default_db_path,
    list_discount_curve_dates,
    list_par_curve_dates,
    load_discount_curve,
    plot_curve_overview,
    plot_discount_factors,
    plot_yield_curve,
)

print("DB:", default_db_path())

In [ ]:
# --- parameters ---
CURVE_ID = "USD_TREASURY_PAR_FRED"
AS_OF = "2026-05-27"  # change me

disc_dates = list_discount_curve_dates(CURVE_ID)
par_dates = list_par_curve_dates(CURVE_ID)
print(f"Discount curve dates ({len(disc_dates)}):", disc_dates[-5:])
print(f"Par curve dates ({len(par_dates)}):", par_dates[-5:])
if AS_OF not in disc_dates:
    raise ValueError(f"{AS_OF!r} not in discount_curve_eod; pick one of the dates above.")

In [ ]:
points, disc_hdr, par_hdr = load_discount_curve(AS_OF, CURVE_ID)

display(pd.DataFrame([disc_hdr]).T.rename(columns={0: "discount_curve_eod"}))
display(pd.DataFrame([par_hdr]).T.rename(columns={0: "par_curve_eod"}))

table = points[
    [
        "tenor",
        "time_years",
        "quoted_rate",
        "zero_rate",
        "discount_factor",
        "instrument_type",
        "fred_series_id",
    ]
].copy()
table["quoted_rate_pct"] = (table["quoted_rate"] * 100).map(lambda x: f"{x:.3f}%")
table["zero_rate_pct"] = (table["zero_rate"] * 100).map(lambda x: f"{x:.3f}%")
table["discount_factor"] = table["discount_factor"].map(lambda x: f"{x:.6f}")
display(table)

In [ ]:
fig = plot_curve_overview(points, discount_header=disc_hdr, par_header=par_hdr)
plt.show()

In [ ]:
fig = plot_yield_curve(points, discount_header=disc_hdr)
plt.show()

fig = plot_discount_factors(points, discount_header=disc_hdr)
plt.show()